# 3. Unbalanced Data: ANOVA versus Mixed Models

*Companion to Chapter 1, pages 23–28.*

The real maize RIL data set has about 4.5% missing plots. RIL-5 has data from only two of the four locations and RIL-51 from three. With unbalanced data the ANOVA and mixed-model analyses part ways: the order of fitting matters, expected mean squares acquire awkward coefficients, standard F-tests become invalid, and some LS-means cannot be estimated at all. The mixed model handles all of this naturally.

In [1]:
# Modules and imports for this section
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from lmmtools.data import load_maize                 # CUSTOM: data loader (Appendix A)
from lmmtools.anova import FactorialANOVA            # CUSTOM: SAS-style Type I/III, EMS, random tests (Appendix C.2)
from lmmtools.design import incidence                # CUSTOM: sparse incidence matrices
from lmmtools.reml import SparseREML                 # CUSTOM: sparse REML with SAS-style SEs (Appendix C.1)
from lmmtools.report import check
import warnings; warnings.filterwarnings("ignore")
pd.set_option("display.width", 100); pd.set_option("display.max_colwidth", 100)
miss = load_maize(balanced=False)
print(miss.shape, "| locations per RIL < 4:", miss.groupby("RIL").location.nunique().loc[lambda s: s < 4].to_dict())

(474, 6) | locations per RIL < 4: {'RIL-5': 2, 'RIL-51': 3}


## Unbalanced data: ANOVA with SAS PROC GLM

**Original SAS code (from the chapter)**

```sas
proc glm data = maize_miss;
 class location rep RIL;
 model mean_height = location rep(location) RIL RIL*location;
 random location rep(location) RIL RIL*location/test;
 lsmeans RIL;
run;
```

statsmodels' `anova_lm` reports 183 df for the interaction, because it counts columns rather than the rank of the design (three location × RIL cells are empty). The custom `FactorialANOVA` uses ranks, giving PROC GLM's 180 df, and builds Type III hypotheses as SAS does (Appendix C.2):

In [2]:
ols_fit = smf.ols("height ~ C(location) + C(location):C(rep) + C(RIL) + C(RIL):C(location)", data=miss).fit()
print("statsmodels anova_lm (typ=1):\n", sm.stats.anova_lm(ols_fit, typ=1).round(4))
terms = {"location": ("location",), "rep(location)": ("location", "rep"), "RIL": ("RIL",), "location*RIL": ("location", "RIL")}
glm = FactorialANOVA(miss, "height", terms)
t1, t3 = glm.type1(), glm.type3()
print("\nType I\n", t1.round(4)); print("\nType III\n", t3.round(4))
print(f"\nR-Square {1 - glm.sse / np.sum((miss.height - miss.height.mean())**2):.6f}   Root MSE {np.sqrt(glm.mse):.6f}")

statsmodels anova_lm (typ=1):
                        df       sum_sq     mean_sq         F  PR(>F)
C(location)           3.0   79345.6274  26448.5425  392.5937  0.0000
C(RIL)               61.0  150084.8234   2460.4069   36.5215  0.0000
C(location):C(rep)    4.0    3895.7505    973.9376   14.4568  0.0000
C(RIL):C(location)  183.0   18294.6297     99.9707    1.4839  0.0024
Residual            225.0   15157.9644     67.3687       NaN     NaN



Type I
                 DF           SS          MS         F  Pr > F
Source                                                       
location         3   79345.6274  26448.5425  392.5937  0.0000
rep(location)    4    3693.2364    923.3091   13.7053  0.0000
RIL             61  150287.3376   2463.7268   36.5708  0.0000
location*RIL   180   18290.6367    101.6146    1.5083  0.0018
Error          225   15157.9644     67.3687       NaN     NaN

Type III
                 DF           SS          MS         F  Pr > F
Source                                                       
location         3   77171.8286  25723.9429  381.8380  0.0000
rep(location)    4    3677.7950    919.4488   13.6480  0.0000
RIL             61  149644.4648   2453.1879   36.4143  0.0000
location*RIL   180   18290.6367    101.6146    1.5083  0.0018
Error          225   15157.9644     67.3687       NaN     NaN

R-Square 0.943181   Root MSE 8.207846


In [3]:
for term, ss in [("location", 79345.6274), ("rep(location)", 3693.2364), ("RIL", 150287.3376), ("location*RIL", 18290.6367)]:
    check(f"Type I SS {term}", t1.loc[term, "SS"], ss, 5e-4)
for term, ss in [("location", 77171.8286), ("rep(location)", 3677.7950), ("RIL", 149644.4648), ("location*RIL", 18290.6367)]:
    check(f"Type III SS {term}", t3.loc[term, "SS"], ss, 5e-4)
check("location*RIL df", t3.loc["location*RIL", "DF"], 180, 0)
check("sum of Type III SS", t3["SS"].iloc[:4].sum(), 248784.7251, 5e-4)
check("Error MS", glm.mse, 67.3687, 5e-5)

PASS  Type I SS location: Python = 79345.6   chapter = 79345.6


PASS  Type I SS rep(location): Python = 3693.24   chapter = 3693.24
PASS  Type I SS RIL: Python = 150287   chapter = 150287
PASS  Type I SS location*RIL: Python = 18290.6   chapter = 18290.6
PASS  Type III SS location: Python = 77171.8   chapter = 77171.8
PASS  Type III SS rep(location): Python = 3677.8   chapter = 3677.8
PASS  Type III SS RIL: Python = 149644   chapter = 149644
PASS  Type III SS location*RIL: Python = 18290.6   chapter = 18290.6
PASS  location*RIL df: Python = 180   chapter = 180
PASS  sum of Type III SS: Python = 248785   chapter = 248785
PASS  Error MS: Python = 67.3687   chapter = 67.3687


Type I and Type III results now differ. Type I sums of squares are sequential (each term adjusted only for the terms before it); Type III sums of squares adjust each term for all the others. With balanced data the terms are orthogonal and the two coincide; with unbalanced data the factors become correlated and the order of fitting matters, so Type III statistics are preferred. The Type III sums of squares no longer add up to the model total (248 784.73 < 266 774.80).

### Expected mean squares and corrected F-tests

The EMS coefficients now depend on the pattern of missing data. PROC GLM computes them; `ems_type3()` does the same by Hartley's method, $\text{coef}(\sigma^2_k) = \operatorname{tr}(Z_k' Q Z_k)/df$ for a sum of squares $y'Qy$:

In [4]:
ems = glm.ems_type3()
print(ems.round(4))

               location  rep(location)     RIL  location*RIL  Error
location       114.4862        57.2431  0.0000        1.8769    1.0
rep(location)    0.0000        57.2500  0.0000        0.0000    1.0
RIL              0.0000        -0.0000  7.6105        1.9149    1.0
location*RIL    -0.0000        -0.0000 -0.0000        1.9348    1.0


In [5]:
check("EMS(location): location*RIL", ems.loc["location", "location*RIL"], 1.8769, 5e-5)
check("EMS(location): rep(location)", ems.loc["location", "rep(location)"], 57.243, 5e-4)
check("EMS(location): location", ems.loc["location", "location"], 114.49, 5e-3)
check("EMS(rep(location))", ems.loc["rep(location)", "rep(location)"], 57.25, 5e-3)
check("EMS(RIL): location*RIL", ems.loc["RIL", "location*RIL"], 1.9149, 5e-5)
check("EMS(RIL): RIL", ems.loc["RIL", "RIL"], 7.6105, 5e-5)
check("EMS(location*RIL)", ems.loc["location*RIL", "location*RIL"], 1.9348, 5e-5)

PASS  EMS(location): location*RIL: Python = 1.87687   chapter = 1.8769
PASS  EMS(location): rep(location): Python = 57.2431   chapter = 57.243
PASS  EMS(location): location: Python = 114.486   chapter = 114.49
PASS  EMS(rep(location)): Python = 57.25   chapter = 57.25
PASS  EMS(RIL): location*RIL: Python = 1.91491   chapter = 1.9149
PASS  EMS(RIL): RIL: Python = 7.61046   chapter = 7.6105
PASS  EMS(location*RIL): Python = 1.93481   chapter = 1.9348


Under $H_0: \sigma^2_G = 0$ the ratio $MS(\text{RIL})/MS(\text{location×RIL})$ has expectation $(\sigma^2_\varepsilon + 1.9149\sigma^2_{GL})/(\sigma^2_\varepsilon + 1.9348\sigma^2_{GL}) \neq 1$, so the simple test is no longer correct. PROC GLM synthesises a denominator with the right expectation and uses Satterthwaite's degrees of freedom:

In [6]:
rt = glm.random_tests()
rt.round(4)

,DF,MS,Error DF,Error MS,F,Pr > F,Error term
Source,,,,,,,
location,3,25723.9429,4.2929,952.5662,27.0049,0.0031,0.9999*MS(rep(location)) + 0.9701*MS(location*RIL) - 0.9699*MS(Error)
rep(location),4,919.4488,225.0000,67.3687,13.6480,0.0000,1.0000*MS(Error)
RIL,61,2453.1879,182.4825,101.2623,24.2261,0.0000,0.9897*MS(location*RIL) + 0.0103*MS(Error)
location*RIL,180,101.6146,225.0000,67.3687,1.5083,0.0018,1.0000*MS(Error)


In [7]:
check("location: error df", rt.loc["location", "Error DF"], 4.2929, 5e-5)
check("location: error MS", rt.loc["location", "Error MS"], 952.566239, 5e-5)
check("location: F", rt.loc["location", "F"], 27.00, 5e-3)
check("location: Pr > F", rt.loc["location", "Pr > F"], 0.0031, 5e-5)
check("RIL: error df", rt.loc["RIL", "Error DF"], 182.48, 5e-3)
check("RIL: error MS", rt.loc["RIL", "Error MS"], 101.262345, 5e-5)
check("RIL: F", rt.loc["RIL", "F"], 24.23, 5e-3)

PASS  location: error df: Python = 4.2929   chapter = 4.2929
PASS  location: error MS: Python = 952.566   chapter = 952.566
PASS  location: F: Python = 27.0049   chapter = 27
PASS  location: Pr > F: Python = 0.00309023   chapter = 0.0031
PASS  RIL: error df: Python = 182.482   chapter = 182.48
PASS  RIL: error MS: Python = 101.262   chapter = 101.262
PASS  RIL: F: Python = 24.2261   chapter = 24.23


### Variance components by the method of moments

The moment estimates must use the unbalanced Type III mean squares and coefficients: $\hat\sigma^2_\varepsilon = 67.37$, $\hat\sigma^2_{GL} = (101.61 - 67.37)/1.9348 = 17.70$, and <mark>$\hat\sigma^2_G = (2453.19 - 1.9149 \times 17.70 - 67.37)/7.6105 = 309.04$</mark><sup>[C6]</sup>. These estimators' statistical properties (and hence their standard errors) are unknown.

> **Correction C6** (p. 26, moments (unbalanced)). Draft: σ²(RIL) = (2539.96 − 1.9149·17.7 − 67.37)/8 = 304.84. Corrected: σ²(RIL) = (2453.19 − 1.9149·17.70 − 67.37)/7.6105 = 309.04. Uses the balanced MS(RIL) and divisor 8; the unbalanced Type III MS(RIL) is 2453.19 and its EMS coefficient 7.6105. 309.04 is close to REML's 309.19.

In [8]:
mom = glm.moments()
print(mom.round(3))
check("sigma2 error (moments)", mom["Error"], 67.37, 5e-3)
check("sigma2 location*RIL (moments)", mom["location*RIL"], 17.70, 5e-3)
check("sigma2 RIL (moments, corrected C6)", mom["RIL"], 309.04, 5e-3)
draft = (2539.96 - 1.9149 * 17.7 - 67.37) / 8
print(f"The draft's arithmetic (balanced MS(RIL) = 2539.96, divisor 8) gives {draft:.2f}")

location         216.370
rep(location)     14.883
RIL              309.038
location*RIL      17.700
Error             67.369
dtype: float64
PASS  sigma2 error (moments): Python = 67.3687   chapter = 67.37
PASS  sigma2 location*RIL (moments): Python = 17.6998   chapter = 17.7
PASS  sigma2 RIL (moments, corrected C6): Python = 309.038   chapter = 309.04
The draft's arithmetic (balanced MS(RIL) = 2539.96, divisor 8) gives 304.84


### LS-means that cannot be estimated

PROC GLM prints "Non-est" for RIL-5 and RIL-51. An LS-mean averages the line's interaction effects with *all* locations, $\text{LSMean}(Y_{\cdot\cdot k}) = \mu + \bar L_\cdot + \overline{B(L)}_{\cdot\cdot} + G_k + \overline{GL}_{\cdot k}$ (eq. 1.7), and with no data for a genotype × location cell that interaction effect is not estimable in the fixed model — so neither is the LS-mean.

In [9]:
ls = glm.lsmeans("RIL")
show = ["RIL-1", "RIL-11", "RIL-12", "RIL-49", "RIL-5", "RIL-50", "RIL-51", "RIL-53"]
print(ls.loc[show, ["LSMEAN", "estimable"]].round(4).fillna("Non-est"))
check("RIL-5 non-estimable", float(ls.loc["RIL-5", "estimable"]), 0, 0)
check("RIL-51 non-estimable", float(ls.loc["RIL-51", "estimable"]), 0, 0)
check("LSMEAN RIL-49", ls.loc["RIL-49", "LSMEAN"], 176.975, 5e-4)
check("LSMEAN RIL-53", ls.loc["RIL-53", "LSMEAN"], 174.425, 5e-4)

         LSMEAN  estimable
RIL                       
RIL-1     182.1       True
RIL-11  182.875       True
RIL-12    185.2       True
RIL-49  176.975       True
RIL-5   Non-est      False
RIL-50  200.275       True
RIL-51  Non-est      False
RIL-53  174.425       True
PASS  RIL-5 non-estimable: Python = 0   chapter = 0
PASS  RIL-51 non-estimable: Python = 0   chapter = 0
PASS  LSMEAN RIL-49: Python = 176.975   chapter = 176.975
PASS  LSMEAN RIL-53: Python = 174.425   chapter = 174.425


## Unbalanced data: mixed-model analysis

**Original SAS code (from the chapter)**

```sas
proc mixed data = ril.maizeril_miss covtest;
class location rep RIL;
model height =/solution;
random location rep(location) RIL RIL*location/solution;
```

In [10]:
d = miss.assign(one=1)
vc = {"location": "0 + C(location)", "rep(location)": "0 + C(location):C(rep)",
      "RIL": "0 + C(RIL)", "location*RIL": "0 + C(RIL):C(location)"}
mm = smf.mixedlm("height ~ 1", d, groups="one", vc_formula=vc).fit(reml=True, method=["bfgs"], gtol=1e-10, maxiter=5000)
Zs = [incidence(miss, f)[0] for f in ["location", ["location", "rep"], "RIL", ["location", "RIL"]]]
model = SparseREML(miss.height, np.ones((len(miss), 1)), Zs, list(vc))
fit = model.fit()
tab = fit.table()
tab["statsmodels"] = list(mm.vcomp[[list(mm.model.exog_vc.names).index(k) for k in vc]]) + [mm.scale]
print(tab.round(4))

               Estimate  Std Error  Z value    Pr Z  statsmodels
location       217.6850   184.6923   1.1786  0.1193     217.6855
rep(location)   15.0610    11.4621   1.3140  0.0944      15.0611
RIL            309.1897    58.5755   5.2785  0.0000     309.1895
location*RIL    17.1466     6.4757   2.6478  0.0041      17.1468
Residual        67.9183     6.4110  10.5940  0.0000      67.9182


In [11]:
sas = {"location": (217.69, 184.69), "rep(location)": (15.0591, 11.4593), "RIL": (309.19, 58.5756),
       "location*RIL": (17.1466, 6.4757), "Residual": (67.9184, 6.4110)}
for k, (est, se) in sas.items():
    check(f"sigma2 {k}", tab.loc[k, "Estimate"], est, 2e-4 * est)
    check(f"SE {k}", tab.loc[k, "Std Error"], se, 5e-4 * se)
    check(f"statsmodels sigma2 {k}", tab.loc[k, "statsmodels"], est, 5e-4 * est)

PASS  sigma2 location: Python = 217.685   chapter = 217.69
PASS  SE location: Python = 184.692   chapter = 184.69
PASS  statsmodels sigma2 location: Python = 217.685   chapter = 217.69
PASS  sigma2 rep(location): Python = 15.061   chapter = 15.0591
PASS  SE rep(location): Python = 11.4621   chapter = 11.4593
PASS  statsmodels sigma2 rep(location): Python = 15.0611   chapter = 15.0591
PASS  sigma2 RIL: Python = 309.19   chapter = 309.19
PASS  SE RIL: Python = 58.5755   chapter = 58.5756
PASS  statsmodels sigma2 RIL: Python = 309.19   chapter = 309.19
PASS  sigma2 location*RIL: Python = 17.1466   chapter = 17.1466
PASS  SE location*RIL: Python = 6.47568   chapter = 6.4757
PASS  statsmodels sigma2 location*RIL: Python = 17.1468   chapter = 17.1466
PASS  sigma2 Residual: Python = 67.9183   chapter = 67.9184
PASS  SE Residual: Python = 6.41101   chapter = 6.411
PASS  statsmodels sigma2 Residual: Python = 67.9182   chapter = 67.9184


The REML estimates are close to, but not the same as, the moment estimates, and they come with asymptotic standard errors and the properties of likelihood estimators under normality.

The BLUPs exist for every line, including RIL-5 and RIL-51: an unobserved genotype × location effect has expectation zero, so the line's value can still be predicted. Shrinkage now varies by line — lines with less data are pulled further toward the mean.

In [12]:
u = pd.Series(fit.blups["RIL"], index=sorted(miss.RIL.unique()))
tbl = pd.DataFrame({"Random effect predictor": u, "mu": fit.beta[0]})
tbl["BLUP"] = tbl["mu"] + tbl["Random effect predictor"]
tbl["LSMEAN"] = ls["LSMEAN"]
tbl["n plots"] = miss.groupby("RIL").size()
pev = pd.Series(model.blup_pev(fit, "RIL"), index=u.index)
tbl["SE pred"] = np.sqrt(pev)
print(tbl.loc[show].round(2).fillna("Non-est"))

        Random effect predictor     mu    BLUP   LSMEAN  n plots  SE pred
RIL-1                      0.39  181.7  182.08    182.1        8     4.13
RIL-11                     1.13  181.7  182.83   182.88        8     4.13
RIL-12                     3.37  181.7  185.06    185.2        8     4.13
RIL-49                    -4.53  181.7  177.16   176.98        8     4.13
RIL-5                    -28.74  181.7  152.96  Non-est        2     6.46
RIL-50                    17.84  181.7  199.54   200.28        8     4.13
RIL-51                     0.53  181.7  182.22  Non-est        3     5.53
RIL-53                    -6.98  181.7  174.71   174.43        8     4.13


In [13]:
for k, v in {"RIL-1": 182.08, "RIL-5": 152.96, "RIL-50": 199.54, "RIL-51": 182.22, "RIL-53": 174.71}.items():
    check(f"BLUP {k}", tbl.loc[k, "BLUP"], v, 6e-3)
check("mu", fit.beta[0], 181.70, 5e-3)

PASS  BLUP RIL-1: Python = 182.084   chapter = 182.08
PASS  BLUP RIL-5: Python = 152.958   chapter = 152.96
PASS  BLUP RIL-50: Python = 199.538   chapter = 199.54
PASS  BLUP RIL-51: Python = 182.221   chapter = 182.22
PASS  BLUP RIL-53: Python = 174.713   chapter = 174.71
PASS  mu: Python = 181.695   chapter = 181.7
